# WAN 2.2 Animate Character Swap — ComfyUI

Kaynak videodan karakter hareketini al, hedef karaktere uygula (pose + face + mask tabanlı).

## Pipeline
```
Input Video → Pose/Face Detection → SAM2 Masking → WAN 2.2 Animate 14B → Character Swap Video
```

## Colab Secrets
- `CF_TUNNEL_TOKEN` — Cloudflare tunnel
- `HF_TOKEN` — HuggingFace

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat

## Custom Node'lar
| Node | Kaynak | Kullanım |
|------|--------|----------|
| ComfyUI-WanVideoWrapper | kijai | Model yükleme, sampling, decode, text encode, animate embeds |
| ComfyUI-KJNodes | kijai | Image resize/concat, mask işleme, utility |
| ComfyUI-VideoHelperSuite | Kosinkadink | Video yükleme/birleştirme/kaydetme |
| ComfyUI-segment-anything-2 | kijai | SAM2 karakter segmentasyonu |
| ComfyUI-WanAnimatePreprocess | kijai | Pose/face detection (ViTPose + YOLO) |

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# ComfyUI
COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

# WanAnimatePreprocess dependency'leri (node'dan önce yükle)
!pip install -q onnxruntime-gpu opencv-python-headless ultralytics

# ComfyUI Manager (node yönetimi)
!pip install -q -U --pre comfyui-manager

# Custom Node'lar
NODES = {
    'ComfyUI-WanVideoWrapper': 'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-segment-anything-2': 'https://github.com/kijai/ComfyUI-segment-anything-2.git',
    'ComfyUI-WanAnimatePreprocess': 'https://github.com/kijai/ComfyUI-WanAnimatePreprocess.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# Node yükleme doğrulama
import importlib, sys
sys.path.insert(0, COMFY_DIR)
preprocess_init = f'{CUSTOM_NODES}/ComfyUI-WanAnimatePreprocess/__init__.py'
if os.path.exists(preprocess_init):
    print('\u2705 WanAnimatePreprocess dosyaları mevcut')
else:
    print('\u274c WanAnimatePreprocess bulunamadı!')

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

WAN 2.2 Animate 14B (FP8) + LoRA'lar + Preprocessor modelleri

In [ ]:
import shutil

from huggingface_hub import hf_hub_download

MODELS_DIR = f'{COMFY_DIR}/models'

def hf_download(repo, filename, dest_dir):
    """HuggingFace'ten dosya indir. Mevcutsa atla."""
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Ba\u015far\u0131s\u0131z: {e}')

# === WAN 2.2 Animate 14B (FP8) ===
print('\U0001f4e5 WAN 2.2 Animate 14B Diffusion Model:')
hf_download(
    'Kijai/WanVideo_comfy_fp8_scaled',
    'Wan22Animate/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors',
    f'{MODELS_DIR}/diffusion_models'
)

# === LoRA'lar ===
print('\n\U0001f4e5 LoRA\'lar:')
# Relight LoRA (aydınlatma uyumu)
hf_download(
    'Kijai/WanVideo_comfy',
    'LoRAs/Wan22_relight/WanAnimate_relight_lora_fp16.safetensors',
    f'{MODELS_DIR}/loras'
)
# LightX2V step distill LoRA (hız)
hf_download(
    'Kijai/WanVideo_comfy',
    'Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors',
    f'{MODELS_DIR}/loras'
)

# === Text Encoder ===
print('\n\U0001f4e5 Text Encoder:')
hf_download(
    'Kijai/WanVideo_comfy',
    'umt5-xxl-enc-bf16.safetensors',
    f'{MODELS_DIR}/text_encoders'
)

# === VAE ===
print('\n\U0001f4e5 VAE:')
hf_download(
    'Kijai/WanVideo_comfy',
    'Wan2_1_VAE_bf16.safetensors',
    f'{MODELS_DIR}/vae'
)
# Workflow JSON 'wan_2.1_vae.safetensors' adıyla referans veriyor
vae_src = f'{MODELS_DIR}/vae/Wan2_1_VAE_bf16.safetensors'
vae_link = f'{MODELS_DIR}/vae/wan_2.1_vae.safetensors'
if os.path.exists(vae_src) and not os.path.exists(vae_link):
    os.symlink(vae_src, vae_link)
    print(f'  \u2705 symlink: wan_2.1_vae.safetensors -> Wan2_1_VAE_bf16.safetensors')

# === CLIP Vision ===
print('\n\U0001f4e5 CLIP Vision:')
hf_download(
    'Comfy-Org/Wan_2.1_ComfyUI_repackaged',
    'split_files/clip_vision/clip_vision_h.safetensors',
    f'{MODELS_DIR}/clip_vision'
)

# === Preprocessor Modelleri (Pose + Detection) ===
print('\n\U0001f4e5 Preprocessor Modelleri:')
PREPROCESS_DIR = f'{CUSTOM_NODES}/ComfyUI-WanAnimatePreprocess/models'
os.makedirs(PREPROCESS_DIR, exist_ok=True)

# ViTPose (Large, wholebody)
hf_download(
    'JunkyByte/easy_ViTPose',
    'onnx/wholebody/vitpose-l-wholebody.onnx',
    PREPROCESS_DIR
)

# YOLO v10m (person detection)
hf_download(
    'Wan-AI/Wan2.2-Animate-14B',
    'process_checkpoint/det/yolov10m.onnx',
    PREPROCESS_DIR
)

# === SAM2 (otomatik indirilir ama path hazırla) ===
print('\n\U0001f4e5 SAM2:')
print('  \u2139\ufe0f sam2.1_hiera_base_plus.safetensors workflow çalıştırılınca otomatik indirilir')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy — hızlı UI

`USE_CLOUDFLARE = True`: Cloudflare tunnel — sabit URL (`comfyui.ersamely.com`)

## Workflow Yükleme
1. ComfyUI açılınca **Load** butonuna tıkla
2. `Wan 2.2 Animate Character Swap.json` dosyasını yükle
3. Input video + referans karakter görseli ayarla
4. **Queue Prompt** ile çalıştır

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False

PORT = 8188

# Önceki process'leri kapat
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# ComfyUI başlat
log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*', '--enable-manager'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

# Hazır olmasını bekle
t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI \u00e7\u00f6kt\u00fc!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')

    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False

    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)